# KEEP Ancestor Roll-Up Ablation: Multi-Parent Policies, Root Choice, and Graph Scale

**Paper:** Elhussein et al., *"KEEP: Integrating Medical Ontologies with Clinical Data for Robust Code Embeddings"*, Conference on Health, Inference, and Learning (CHIL), PMLR 287:1–19, 2025 ([paper](https://proceedings.mlr.press/v287/elhussein25a.html)).

## Background and PyHealth Design

In **Appendix A.1.1 (Graph Creation)**, KEEP constructs a condition hierarchy from the OMOP `CONCEPT` and `CONCEPT_ANCESTOR` tables by selecting valid standard `Condition` concepts within five minimum levels of separation (`max_depth = 5`) from the SNOMED root concept **Disease** (`concept_id = 4274025`).

In **Appendix A.4 (Co-occurrence matrix)**, rather than discarding diagnosis codes more than five levels away from the root, KEEP rolls each deep code up to its ancestor concepts present in the depth-limited graph. When a concept has multiple parents in the ontology, KEEP adopts a dense roll-up that maps the code across its parent branches.

Because the paper leaves exact ancestor selection open when rolling up across multi-parent branches, PyHealth exposes the design choices as explicit **PyHealth policies** on `AncestorRollupProcessor` and `build_keep_omop_ancestor_rollup()`:

- `target_policy="first_retained"`: stops upward traversal independently on each branch at the first eligible ancestor within `max_depth`.
- `target_policy="all_retained"`: continues upward traversal past the first eligible ancestor and collects every distinct eligible strict ancestor.
- `include_root=False` vs. `True`: controls whether the hierarchy root itself is eligible as a roll-up target.
- `path_source="direct"` vs. `"closure"`: traverses direct `"Is a"` edges alone, or augments them with minimum-separation-one `CONCEPT_ANCESTOR` pairs over the uncut subtree.

The decoded path graph uses PyTorch Geometric. From the PyHealth checkout, install the `graph` extra in the notebook kernel environment with `pip install -e '.[graph]'`.

> **Scope note:** This notebook focuses on ontology structure, multi-parent roll-up policies, root choice, and DAG scaling using self-contained synthetic data and public PyHealth APIs. Patient-history co-occurrence counting and association measures are reserved for a separate statistical study.

In [ ]:
import os
import tempfile
import time
from pathlib import Path

import polars as pl

from pyhealth.graph import KnowledgeGraph
from pyhealth.medcode import DISEASE_CONCEPT_ID, OMOPVocabulary
from pyhealth.processors import AncestorRollupProcessor, UnmappedCodeError
from pyhealth.tasks import build_keep_omop_ancestor_rollup

## 1. Multi-Parent Diamond Hierarchy: Policies, Root Inclusion, and Path Graph

Medical ontologies such as SNOMED CT are polyhierarchical directed acyclic graphs (DAGs), not strict trees: a clinical concept can inherit from multiple parent categories whose ancestor paths later converge.

Below we construct a hierarchy rooted at `Disease` (depth 0) with a diamond under `Cardiovascular_Disease` (depth 1):

```text
Disease (depth 0)
 ├── Cardiovascular_Disease (depth 1)
 │    ├── Ischemic_Heart_Disease (depth 2)
 │    │    └── Acute_MI (depth 3)
 │    │         └── Post_MI_AV_Block (depth 4)  <-- deep source (two parents)
 │    └── Conduction_Disorder (depth 2)          /
 │         └── AV_Block (depth 3) --------------'
 └── Respiratory_Disease (depth 1)
```

At a fixed cutoff (`max_depth = 3`), concepts at depths 1–3 are retained in the target vocabulary (along with `Disease` at depth 0 when `include_root=True`), while `Post_MI_AV_Block` (depth 4) must roll up.

Key semantics demonstrated below:
1. **Branch-local stopping (`first_retained`):** Each upward branch from `Post_MI_AV_Block` stops as soon as it reaches its first eligible ancestor (`Acute_MI` on one branch, `AV_Block` on the other). Ancestors higher up (`Ischemic_Heart_Disease`, `Conduction_Disorder`, `Cardiovascular_Disease`, `Disease`) are not emitted.
2. **Full ancestor collection (`all_retained`):** Upward traversal continues past `Acute_MI` and `AV_Block` through `Ischemic_Heart_Disease`, `Conduction_Disorder`, `Cardiovascular_Disease`, and (when `include_root=True`) `Disease`.
3. **Target deduplication:** Although both branches of the diamond converge on `Cardiovascular_Disease` (and `Disease`), `RollupResult.targets` deduplicates targets and orders them lexicographically.
4. **Root inclusion (`include_root`):** Under `all_retained`, setting `include_root=True` adds `Disease` to the targets of `Post_MI_AV_Block`. Under `first_retained`, `Disease` is only emitted when a branch reaches the root without encountering an intermediate retained concept (for example at `max_depth = 0`).

In [ ]:
diamond_triples = [
    ("Disease", "is_a", "Cardiovascular_Disease"),
    ("Cardiovascular_Disease", "is_a", "Ischemic_Heart_Disease"),
    ("Cardiovascular_Disease", "is_a", "Conduction_Disorder"),
    ("Ischemic_Heart_Disease", "is_a", "Acute_MI"),
    ("Conduction_Disorder", "is_a", "AV_Block"),
    ("Acute_MI", "is_a", "Post_MI_AV_Block"),
    ("AV_Block", "is_a", "Post_MI_AV_Block"),
    ("Disease", "is_a", "Respiratory_Disease"),
]

diamond_kg = KnowledgeGraph(triples=diamond_triples)
diamond_domains = {code: "Condition" for code in diamond_kg.entity2id}

CUTOFF = 3
print(f"Fixed cutoff max_depth={CUTOFF} on diamond hierarchy:")
for policy in ("first_retained", "all_retained"):
    for include_root in (False, True):
        proc = AncestorRollupProcessor(
            knowledge_graph=diamond_kg,
            domain_by_code=diamond_domains,
            hierarchy_relation="is_a",
            root="Disease",
            target_policy=policy,
            include_root=include_root,
            max_depth=CUTOFF,
            on_unmapped="empty",
        )
        res_deep, res_kept = proc.rollup_codes(["Post_MI_AV_Block", "Acute_MI"])
        print(
            f"  policy={policy:<14} include_root={str(include_root):<5} | "
            f"source={res_deep.source} (depth {res_deep.source_depth}, {res_deep.reason}) "
            f"-> targets={list(res_deep.targets)}"
        )
        assert res_kept.reason == "retained" and res_kept.targets == ("Acute_MI",)

# Decode one projected path graph for Post_MI_AV_Block under all_retained (include_root=False)
all_retained_proc = AncestorRollupProcessor(
    knowledge_graph=diamond_kg,
    domain_by_code=diamond_domains,
    hierarchy_relation="is_a",
    root="Disease",
    target_policy="all_retained",
    include_root=False,
    max_depth=CUTOFF,
    on_unmapped="empty",
)
path_graph = all_retained_proc.process(["Post_MI_AV_Block"])
decoded_nodes = [
    all_retained_proc.id2entity[int(nid)] for nid in path_graph.node_ids.tolist()
]
decoded_edges = [
    (
        decoded_nodes[int(path_graph.edge_index[0, i])],
        decoded_nodes[int(path_graph.edge_index[1, i])],
    )
    for i in range(path_graph.num_edges)
]
seed_nodes = [
    code for code, is_seed in zip(decoded_nodes, path_graph.seed_mask.tolist()) if is_seed
]
target_nodes = [
    code for code, is_target in zip(decoded_nodes, path_graph.target_mask.tolist()) if is_target
]

print("\nDecoded projected path graph (all_retained, include_root=False):")
print("  Nodes:  ", decoded_nodes)
print("  Seeds:  ", seed_nodes)
print("  Targets:", target_nodes)
print("  Edges (parent -> child):")
for parent, child in decoded_edges:
    print(f"    {parent} -> {child}")

## 2. Root Choice, Domain & DAG Requirements, and the OMOP `Disease` Hierarchy

### 2.1 Broader Root and New Branch at the Same Cutoff

Suppose we embed our `Disease` hierarchy under a broader synthetic root `Clinical_Finding` (`Clinical_Finding -> Disease`) and attach a sibling branch (`Clinical_Finding -> Finding_of_Trunk -> Chest_Pain -> Rest_Angina_Pain`).

Holding the cutoff fixed at `max_depth = 3` and using `AncestorRollupProcessor`:
- **When `root="Disease"`:**
  - `Disease` is at depth 0, so `Acute_MI` and `AV_Block` sit at depth 3 (retained), and `Post_MI_AV_Block` (depth 4) rolls up to `['AV_Block', 'Acute_MI']` under `first_retained`.
  - Concepts on the sibling branch (`Rest_Angina_Pain`) have no directed path from `Disease` and fail with `reason="unreachable"`.
- **When `root="Clinical_Finding"` at the same cutoff (`max_depth = 3`):**
  - **Reachability changes:** `Rest_Angina_Pain` is now reachable at depth 3 from `Clinical_Finding` and is retained by identity (`reason="retained"`).
  - **Eligibility shifts:** Every concept under `Disease` shifts one level deeper (`Disease` is now depth 1). `Acute_MI` and `AV_Block` move from depth 3 to depth 4 and are **no longer eligible targets** at `max_depth = 3`. `Acute_MI` (now depth 4) rolls up to `['Ischemic_Heart_Disease']`. `Post_MI_AV_Block` (now depth 5) rolls up across both branches to `['Conduction_Disorder', 'Ischemic_Heart_Disease']`.

### 2.2 `Condition` Domain and DAG Requirements

- **Condition domain requirement:** Following KEEP Appendix A.1.1, only concepts in the `"Condition"` domain can serve as roll-up sources or eligible targets. Known concepts from other domains (e.g., `"Observation"` or `"Procedure"`) receive `reason="outside_domain"`.
- **DAG requirement:** Upward ancestor roll-up requires a directed acyclic graph (DAG). Self-edges (`A -> A`) and directed cycles (`A -> B -> A`) are rejected during `AncestorRollupProcessor` construction with a `ValueError`.

In [ ]:
broader_triples = diamond_triples + [
    ("Clinical_Finding", "is_a", "Disease"),
    ("Clinical_Finding", "is_a", "Finding_of_Trunk"),
    ("Finding_of_Trunk", "is_a", "Chest_Pain"),
    ("Chest_Pain", "is_a", "Rest_Angina_Pain"),
    ("Cardiovascular_Disease", "is_a", "ECG_Observation"),
]

broader_kg = KnowledgeGraph(triples=broader_triples)
broader_domains = {
    code: ("Observation" if code == "ECG_Observation" else "Condition")
    for code in broader_kg.entity2id
}

query_codes = [
    "Acute_MI",
    "Post_MI_AV_Block",
    "Rest_Angina_Pain",
    "ECG_Observation",
    "Unknown_Concept",
]

for chosen_root in ("Disease", "Clinical_Finding"):
    proc = AncestorRollupProcessor(
        knowledge_graph=broader_kg,
        domain_by_code=broader_domains,
        hierarchy_relation="is_a",
        root=chosen_root,
        target_policy="first_retained",
        include_root=False,
        max_depth=CUTOFF,
        on_unmapped="empty",
    )
    print(f"=== Root: {chosen_root!r} (max_depth={CUTOFF}, first_retained) ===")
    print(f"  Retained eligible targets: {sorted(proc.retained_codes)}")
    for res in proc.rollup_codes(query_codes):
        print(
            f"  {res.source:<20} depth={str(res.source_depth):<4} "
            f"reason={res.reason:<14} targets={list(res.targets)}"
        )
    print()

# Demonstrate DAG validation: a directed cycle is rejected at construction time
cyclic_kg = KnowledgeGraph(
    triples=[
        ("Disease", "is_a", "Node_A"),
        ("Node_A", "is_a", "Node_B"),
        ("Node_B", "is_a", "Node_A"),
    ]
)
try:
    AncestorRollupProcessor(
        knowledge_graph=cyclic_kg,
        domain_by_code={c: "Condition" for c in cyclic_kg.entity2id},
        hierarchy_relation="is_a",
        root="Disease",
        target_policy="first_retained",
        include_root=False,
    )
except ValueError as exc:
    print("DAG check rejected cycle as expected:", exc)

### 2.3 OMOP Example Rooted at `Disease` (`DISEASE_CONCEPT_ID = 4274025`)

For OMOP vocabularies, `build_keep_omop_ancestor_rollup()` pins the root to SNOMED **Disease** (`DISEASE_CONCEPT_ID = 4274025`) and uses authoritative minimum separations from `CONCEPT_ANCESTOR` (`depth_measure="min"`, Appendix A.1.1) over the uncut Condition subtree. Below we create a synthetic Athena bundle with a depth-6 multi-parent disease concept (`1006`) and run `build_keep_omop_ancestor_rollup()` at KEEP's default `max_depth = 5`.

In [ ]:
def write_synthetic_omop_bundle(folder: Path) -> None:
    """Write a small Athena-format OMOP bundle rooted at Disease (4274025)."""
    concepts = [
        (700, "Clinical finding", "Condition", "SNOMED", "S", "404684003", ""),
        (DISEASE_CONCEPT_ID, "Disease", "Condition", "SNOMED", "S", "64572001", ""),
        (1001, "Disorder of cardiovascular system", "Condition", "SNOMED", "S", "49601007", ""),
        (1002, "Heart disease", "Condition", "SNOMED", "S", "56265001", ""),
        (1003, "Structural heart disease", "Condition", "SNOMED", "S", "100003", ""),
        (1004, "Myocardial disease", "Condition", "SNOMED", "S", "57809008", ""),
        (1005, "Acute myocardial infarction", "Condition", "SNOMED", "S", "57054005", ""),
        (2005, "Cardiac conduction block", "Condition", "SNOMED", "S", "200005", ""),
        (1006, "Post-infarction conduction block", "Condition", "SNOMED", "S", "100006", ""),
        (3000, "Fever (under Clinical finding only)", "Condition", "SNOMED", "S", "386661006", ""),
        (6000, "Resting heart rate observation", "Observation", "SNOMED", "S", "600000", ""),
    ]
    relationships = [
        (DISEASE_CONCEPT_ID, 700, "Is a", ""),
        (1001, DISEASE_CONCEPT_ID, "Is a", ""),
        (1002, 1001, "Is a", ""),
        (1003, 1002, "Is a", ""),
        (1004, 1003, "Is a", ""),
        (1005, 1004, "Is a", ""),
        (2005, 1004, "Is a", ""),
        (1006, 1005, "Is a", ""),
        (1006, 2005, "Is a", ""),
        (3000, 700, "Is a", ""),
    ]
    # Shortest/longest separations from Disease (4274025) and intermediate ancestors
    ancestors = [
        (DISEASE_CONCEPT_ID, DISEASE_CONCEPT_ID, 0, 0),
        (DISEASE_CONCEPT_ID, 1001, 1, 1),
        (DISEASE_CONCEPT_ID, 1002, 2, 2),
        (DISEASE_CONCEPT_ID, 1003, 3, 3),
        (DISEASE_CONCEPT_ID, 1004, 4, 4),
        (DISEASE_CONCEPT_ID, 1005, 5, 5),
        (DISEASE_CONCEPT_ID, 2005, 5, 5),
        (DISEASE_CONCEPT_ID, 1006, 6, 6),
        (1005, 1006, 1, 1),
        (2005, 1006, 1, 1),
        (1004, 1006, 2, 2),
    ]

    concept_header = [
        "concept_id", "concept_name", "domain_id", "vocabulary_id",
        "concept_class_id", "standard_concept", "concept_code", "invalid_reason",
    ]
    concept_lines = ["\t".join(concept_header)] + [
        "\t".join(str(x) for x in (cid, name, dom, voc, "Clinical Finding", std, code, inv))
        for cid, name, dom, voc, std, code, inv in concepts
    ]
    (folder / "CONCEPT.csv").write_text("\n".join(concept_lines) + "\n", encoding="utf-8")

    rel_header = ["concept_id_1", "concept_id_2", "relationship_id", "invalid_reason"]
    rel_lines = ["\t".join(rel_header)] + [
        "\t".join(str(x) for x in row) for row in relationships
    ]
    (folder / "CONCEPT_RELATIONSHIP.csv").write_text("\n".join(rel_lines) + "\n", encoding="utf-8")

    anc_header = [
        "ancestor_concept_id", "descendant_concept_id",
        "min_levels_of_separation", "max_levels_of_separation",
    ]
    anc_lines = ["\t".join(anc_header)] + [
        "\t".join(str(x) for x in row) for row in ancestors
    ]
    (folder / "CONCEPT_ANCESTOR.csv").write_text("\n".join(anc_lines) + "\n", encoding="utf-8")


with tempfile.TemporaryDirectory() as tmp_dir:
    write_synthetic_omop_bundle(Path(tmp_dir))
    omop_vocab = OMOPVocabulary(tmp_dir)
    omop_proc = build_keep_omop_ancestor_rollup(
        omop_vocab,
        target_policy="first_retained",
        include_root=False,
        max_depth=5,
        on_unmapped="empty",
        path_source="direct",
    )
    print(f"OMOP hierarchy root: {omop_proc.provenance['hierarchy_root']} (Disease={DISEASE_CONCEPT_ID})")
    print(f"Retained targets at max_depth=5: {sorted(omop_proc.retained_codes)}")
    for res in omop_proc.rollup_codes(["1005", "1006", "3000", "6000", "999999"]):
        print(
            f"  code={res.source:<8} depth={str(res.source_depth):<5} "
            f"reason={res.reason:<14} targets={list(res.targets)}"
        )

## 3. Layered DAG Width Scaling and Opt-In Local Athena Benchmark

### 3.1 Synthetic Layered DAG Scaling

To see how DAG size affects processor construction and code roll-up at a fixed depth cutoff, we generate a layered DAG with 6 layers below the root (`depths 1..6`):
- **Fixed settings:** `max_depth = 5` (layers 1..5 eligible, layer 6 deep sources), `path_source = "direct"`, `target_policy = "first_retained"`, `include_root = False`, and `1,000` queries sampled from layer 6.
- **Scaling axis:** we widen each layer (`width_per_layer` in `[25, 50, 100, 200, 400]`), where every node in layer $d \ge 2$ connects to 2 parents in layer $d - 1$.

In [ ]:
def build_layered_dag(width_per_layer: int, num_layers: int = 6):
    """Build a layered DAG where each node in layer >= 2 has 2 parents in layer d-1."""
    root = "ROOT"
    triples = []
    domains = {root: "Condition"}
    prev_layer = [f"L1_{i:04d}" for i in range(width_per_layer)]
    for node in prev_layer:
        domains[node] = "Condition"
        triples.append((root, "is_a", node))

    for layer_idx in range(2, num_layers + 1):
        curr_layer = [f"L{layer_idx}_{i:04d}" for i in range(width_per_layer)]
        for i, node in enumerate(curr_layer):
            domains[node] = "Condition"
            p1 = prev_layer[i % width_per_layer]
            p2 = prev_layer[(i + 1) % width_per_layer]
            triples.append((p1, "is_a", node))
            if p2 != p1:
                triples.append((p2, "is_a", node))
        prev_layer = curr_layer

    return KnowledgeGraph(triples=triples), domains, prev_layer


QUERY_COUNT = 1000
FIXED_CUTOFF = 5
FIXED_POLICY = "first_retained"
FIXED_PATH_MODE = "direct"

print(
    f"Layered DAG scaling (cutoff={FIXED_CUTOFF}, path_source={FIXED_PATH_MODE!r}, "
    f"policy={FIXED_POLICY!r}, queries={QUERY_COUNT}):"
)
print(
    f"{'width':>7} {'nodes':>8} {'edges':>8} {'eligible':>10} "
    f"{'build_ms':>10} {'map_ms':>10}"
)
for width in (25, 50, 100, 200, 400):
    kg, domains, deepest_nodes = build_layered_dag(width, num_layers=6)
    queries = [deepest_nodes[i % len(deepest_nodes)] for i in range(QUERY_COUNT)]

    t0 = time.perf_counter()
    proc = AncestorRollupProcessor(
        knowledge_graph=kg,
        domain_by_code=domains,
        hierarchy_relation="is_a",
        root="ROOT",
        target_policy=FIXED_POLICY,
        include_root=False,
        max_depth=FIXED_CUTOFF,
        path_source=FIXED_PATH_MODE,
        on_unmapped="error",
    )
    build_ms = (time.perf_counter() - t0) * 1000.0

    t1 = time.perf_counter()
    mapped = proc.rollup_codes(queries)
    map_ms = (time.perf_counter() - t1) * 1000.0

    assert len(mapped) == QUERY_COUNT and all(len(r.targets) == 2 for r in mapped)
    print(
        f"{width:>7} {kg.num_entities:>8} {kg.num_triples:>8} "
        f"{len(proc.retained_codes):>10} {build_ms:>10.2f} {map_ms:>10.2f}"
    )

### 3.2 Opt-In Local Athena Vocabulary Check

Because Athena vocabularies are licensed and not redistributed with PyHealth, the cell below is opt-in. Set `PYHEALTH_ATHENA_ROOT` in your environment (or set `ATHENA_ROOT` directly) to an unzipped Athena directory to report the vocabulary release and aggregate Disease graph statistics.

In [ ]:
ATHENA_ROOT = os.environ.get("PYHEALTH_ATHENA_ROOT", "").strip()

if ATHENA_ROOT and Path(ATHENA_ROOT).expanduser().is_dir():
    athena_path = Path(ATHENA_ROOT).expanduser()
    vocab_csv = athena_path / "VOCABULARY.csv"
    release = "unknown"
    if vocab_csv.is_file():
        vocab_df = pl.read_csv(
            vocab_csv, separator="\t", quote_char=None, infer_schema=False
        )
        row = vocab_df.filter(pl.col("vocabulary_id") == "None")
        if row.height:
            release = row["vocabulary_version"][0]

    t0 = time.perf_counter()
    vocab = OMOPVocabulary(athena_path)
    athena_proc = build_keep_omop_ancestor_rollup(
        vocab,
        target_policy="first_retained",
        include_root=False,
        max_depth=5,
        on_unmapped="empty",
        path_source="direct",
    )
    elapsed_s = time.perf_counter() - t0
    print(f"Athena vocabulary release: {release}")
    print(f"Disease subtree nodes:     {athena_proc.knowledge_graph.num_entities}")
    print(f"Direct 'Is a' edges:       {athena_proc.knowledge_graph.num_triples}")
    print(f"Eligible targets (d<=5):   {len(athena_proc.retained_codes)}")
    print(f"Factory build time:        {elapsed_s:.2f}s")
else:
    print(
        "Opt-in local Athena check skipped (set PYHEALTH_ATHENA_ROOT to an "
        "unzipped Athena directory to report release and aggregate statistics)."
    )